# Operating Window - which combined X range makes BOTH Viscosity and Kappa most capable (Cpk)

**Idea:** Capability -> Operating Window -> Critical X Combination.
We score **two Y together** - Viscosity (`CAP_SPEC` = 700-800, target 750) and Kappa (`CAP_SPEC2` = 9-11, target 10) -
inside boxes made by quantile-binning several X at once, and find the combination with the best **Cpk_min**.

`Cpk_min = min(Cpk_viscosity, Cpk_kappa)` -> a window only wins if it is good for **both** Y (they often trade off).

| Step | What | Why |
|---|---|---|
| 0 | Baseline Cpk (both Y) over all batches | the number any window must beat |
| 0.5 | Baseline Cpk by digester (D1-D4) | is the spread a "which pot" effect? |
| 1 | Per-X Cpk by bin (both Y) | quick look: does one X alone move Cpk? |
| 2 | Grid search of X combinations | the Critical X Combination (ranked by Cpk_min) |
| 3 | Inside vs outside the best window | does the window really win for both Y? |

**X tested (`CAP_X`, 5):** P-factor overshoot, Alkali conc 1, Prehyd minutes, Chip weight, **Digester** (categorical,
split by its 4 pots instead of quantiles). **Edit `config.py`** (section *Capability / Operating Window*) to change
X, specs, bins, combo size, min N.

`Cpk = min((USL-mean), (mean-LSL)) / 3sd`. It assumes each Y is roughly normal in the box; skew is flagged.
Boxes with `N < CAP_MIN_N` are dropped (small-sample Cpk is noise). No PI access - reads batch_master.csv.
Trip outliers of X are dropped first in `load_golden()` (`config.X_VALID_RANGE`: P-factor overshoot <= 134,
Prehyd minutes <= 100), so box edges are not stretched by runaway batches.

In [153]:
import importlib
import pandas as pd

import config, data, golden, capability as cap
for mod in (config, data, golden, cap):
    importlib.reload(mod)

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 40)

## Step 0 - Baseline capability
Cpk of **Viscosity and Kappa** over every batch, plus `Cpk_min`. This is the bar the operating window must beat.

In [154]:
df = golden.load_golden()
_ = cap.baseline(df)

golden data: 740 batches with Kappa + Viscosity (2026-05-09 .. 2026-10-14), golden 299 (40%)
baseline: N=740  Viscosity mean=776.2 Cpk=0.19 (spec 700-800)  |  Kappa mean=10.1 Cpk=0.32 (spec 9-11)  |  Cpk_min=0.19


## Step 0.5 - Capability by digester
Cpk of Viscosity and Kappa per digester (D1-D4) vs overall. If the pots differ a lot, the spread is a "which pot"
effect and each pot may need its own window; if they are close, the pooled analysis below is fine.

In [155]:
dg = cap.baseline_by_digester(df)
display(dg.round({"mean": 1, "std": 1, "cpk": 2, "cp": 2}))

baseline by digester (Viscosity 700-800, Kappa 9-11):


,digester,n,visc_mean,cp_visc,cpk_visc,kappa_mean,cp_kappa,cpk_kappa,cpk_min
0,1,209,771.73,0.45,0.25,9.87,0.36,0.32,0.25
1,2,211,774.60,0.36,0.18,10.22,0.40,0.31,0.18
2,3,218,781.91,0.38,0.14,10.17,0.37,0.31,0.14
3,4,102,776.83,0.50,0.23,10.36,0.36,0.23,0.23
4,ALL,740,776.25,0.40,0.19,10.13,0.37,0.32,0.19


## Step 1 - One X at a time
Cpk of **Viscosity and Kappa** per quantile bin of each X. If a single X already moves either Cpk a lot, it will
matter in the combo. Digester is split by its 4 pots (not quantiles).

In [156]:
for x, tbl in cap.bin_tables(df).items():
    display(tbl.round({"mean": 1, "std": 1, "cpk": 2, "cp": 2}))

,x,range,n,visc_mean,cp_visc,cpk_visc,kappa_mean,cp_kappa,cpk_kappa,cpk_min
0,P-factor overshoot,25.92-57.35,185,795.02,0.36,0.04,10.27,0.37,0.27,0.04
1,P-factor overshoot,57.35-68.86,185,772.66,0.44,0.24,10.19,0.39,0.32,0.24
2,P-factor overshoot,68.86-78.87,185,772.34,0.46,0.26,10.10,0.38,0.34,0.26
3,P-factor overshoot,78.87-133.86,185,764.97,0.42,0.30,9.96,0.35,0.34,0.30


,x,range,n,visc_mean,cp_visc,cpk_visc,kappa_mean,cp_kappa,cpk_kappa,cpk_min
0,Alkali conc 1,34.78-36.23,185,803.40,0.38,-0.03,10.14,0.38,0.33,-0.03
1,Alkali conc 1,36.23-36.82,185,779.98,0.51,0.20,9.99,0.38,0.38,0.20
2,Alkali conc 1,36.82-37.20,185,762.62,0.47,0.35,10.11,0.35,0.31,0.31
3,Alkali conc 1,37.20-38.13,185,758.99,0.44,0.36,10.27,0.37,0.27,0.27


,x,range,n,visc_mean,cp_visc,cpk_visc,kappa_mean,cp_kappa,cpk_kappa,cpk_min
0,Prehyd minutes,56.08-72.33,185,763.64,0.47,0.34,9.82,0.40,0.33,0.33
1,Prehyd minutes,72.33-79.66,185,772.82,0.49,0.26,10.08,0.35,0.32,0.26
2,Prehyd minutes,79.66-84.46,185,795.90,0.38,0.03,10.24,0.36,0.27,0.03
3,Prehyd minutes,84.46-110.00,185,772.63,0.38,0.21,10.36,0.42,0.26,0.21


,x,range,n,visc_mean,cp_visc,cpk_visc,kappa_mean,cp_kappa,cpk_kappa,cpk_min
0,Chip weight,50.01-82.11,185,776.49,0.45,0.21,10.22,0.36,0.28,0.21
1,Chip weight,82.11-88.41,185,777.66,0.39,0.17,10.08,0.38,0.35,0.17
2,Chip weight,88.41-93.61,185,774.63,0.42,0.21,9.99,0.36,0.35,0.21
3,Chip weight,93.61-112.58,185,776.22,0.36,0.17,10.23,0.38,0.30,0.17


,x,range,n,visc_mean,cp_visc,cpk_visc,kappa_mean,cp_kappa,cpk_kappa,cpk_min
0,Digester,1,209,771.73,0.45,0.25,9.87,0.36,0.32,0.25
1,Digester,2,211,774.60,0.36,0.18,10.22,0.40,0.31,0.18
2,Digester,3,218,781.91,0.38,0.14,10.17,0.37,0.31,0.14
3,Digester,4,102,776.83,0.50,0.23,10.36,0.36,0.23,0.23


,x,range,n,visc_mean,cp_visc,cpk_visc,kappa_mean,cp_kappa,cpk_kappa,cpk_min
0,P-temp SP,162.00-163.00,317,792.74,0.38,0.06,10.40,0.38,0.23,0.06
1,P-temp SP,163.00-164.00,99,756.85,0.51,0.44,10.14,0.38,0.33,0.33
2,P-temp SP,164.00-166.00,115,755.72,0.54,0.48,9.84,0.45,0.38,0.38


สรุป cpk ที่ดี
P-factor overshoot	62.6-76.4, Alkali conc 1	37.1-38.8	


## Step 2 - Critical X Combination (grid search)
Every combination of `CAP_COMBO_SIZE` X, each numeric X split into `CAP_BINS` quantile bins (Digester split by pot).
Ranked by **Cpk_min** (the window must be good for both Y), boxes with `N >= CAP_MIN_N` only.
Top row = the window where both Viscosity and Kappa are most capable at once.

In [157]:
res = cap.search(df)
display(cap.summary(res, 12))

search: 76 boxes with N >= 15  (combo size 4, 4 bins/X, ranked by cpk_min)


,combo,P-factor overshoot,Alkali conc 1,Prehyd minutes,Chip weight,cp_visc,cp_kappa,Digester,P-temp SP,n,cpk_visc,cpk_kappa,cpk_min
0,Alkali conc 1 x Prehyd minutes x Dig...,NaN,36.29-36.85,56.08-73.23,NaN,0.712420,0.789442,3,164.00-166.00,21,0.56,0.70,0.56
1,P-factor overshoot x Alkali conc 1 x...,78.87-133.86,36.23-36.82,56.08-72.33,NaN,0.731871,0.761046,3,NaN,17,0.55,0.69,0.55
2,P-factor overshoot x Prehyd minutes ...,77.61-133.86,NaN,56.08-73.23,NaN,0.681588,0.811039,3,164.00-166.00,27,0.55,0.71,0.55
3,Prehyd minutes x Chip weight x Diges...,NaN,NaN,56.08-73.23,79.48-86.81,0.597609,0.686294,3,164.00-166.00,21,0.49,0.60,0.49
4,P-factor overshoot x Alkali conc 1 x...,77.61-133.86,36.29-36.85,56.08-73.23,NaN,0.686559,0.619308,NaN,164.00-166.00,22,0.47,0.57,0.47
5,P-factor overshoot x Alkali conc 1 x...,77.61-133.86,36.29-36.85,NaN,NaN,0.791088,0.565479,3,164.00-166.00,21,0.62,0.46,0.46
6,P-factor overshoot x Chip weight x D...,77.61-133.86,NaN,NaN,86.81-92.11,1.146682,0.575463,3,164.00-166.00,15,1.00,0.43,0.43
7,Alkali conc 1 x Prehyd minutes x Chi...,NaN,36.29-36.85,56.08-73.23,86.81-92.11,0.908951,0.583510,NaN,164.00-166.00,21,0.67,0.43,0.43
8,Alkali conc 1 x Prehyd minutes x Chi...,NaN,37.21-38.13,84.79-110.00,92.11-110.59,0.546614,0.400391,NaN,162.00-163.00,17,0.52,0.40,0.40
9,Alkali conc 1 x Prehyd minutes x Chi...,NaN,37.21-38.13,84.79-110.00,92.11-110.59,0.421627,0.502955,NaN,163.00-164.00,15,0.41,0.39,0.39


## Step 3 - Best window: inside vs outside
Cpk / mean / std / N for **both Y** inside the best box vs the rest. A real window has higher Cpk_min inside.

In [158]:
_ = cap.report(df)

search: 76 boxes with N >= 15  (combo size 4, 4 bins/X, ranked by cpk_min)
best window: Alkali conc 1 36.29-36.85, Prehyd minutes 56.08-73.23, Digester 3, P-temp SP 164.00-166.00
           n  visc_mean   cp_visc  cpk_visc  kappa_mean  cp_kappa  cpk_kappa  cpk_min
inside    22      762.2  0.707860      0.53         9.9  0.758778       0.70     0.53
outside  509      778.7  0.383055      0.16        10.2  0.374191       0.28     0.16
